<a href="https://colab.research.google.com/github/HazCodesLots/Nueral-Image-Caption-Generator/blob/main/ESRGAN-VideoUpscaler.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install torch torchvision basicsr facexlib gfpgan
!git clone https://github.com/xinntao/Real-ESRGAN
%cd Real-ESRGAN
!python setup.py develop
!sed -i 's/functional_tensor/functional/g' /usr/local/lib/python3.11/dist-packages/basicsr/data/degradations.py
!wget https://github.com/xinntao/Real-ESRGAN/releases/download/v0.1.0/RealESRGAN_x4plus.pth -P weights
import os
import cv2
from google.colab import files
import shutil
import time
import glob
import matplotlib.pyplot as plt
from moviepy.editor import VideoFileClip, AudioFileClip

In [ ]:
upload_folder = 'upload'
result_folder = 'results'

for folder in [upload_folder, result_folder]:
    if os.path.exists(folder):
        shutil.rmtree(folder)
    os.makedirs(folder)

uploaded = files.upload()

for filename in uploaded.keys():
    dst_path = os.path.join(upload_folder, filename)
    print(f'Moving {filename} to {dst_path}')
    shutil.move(filename, dst_path)

video_files = [f for f in os.listdir(upload_folder) if f.lower().endswith((".mp4", ".mov", ".avi", ".mkv"))]
if not video_files:
    raise ValueError(" No video file found in upload/. Please upload a .mp4 or similar.")

video_path = os.path.join(upload_folder, video_files[0])

cap = cv2.VideoCapture(video_path)
frame_idx = 0
while True:
    ret, frame = cap.read()
    if not ret:
        break
    frame_path = os.path.join(upload_folder, f"frame_{frame_idx:05d}.png")
    cv2.imwrite(frame_path, frame)
    frame_idx += 1
cap.release()

print(f" Extracted {frame_idx} frames to '{upload_folder}'")


In [ ]:
for f in os.listdir("upload"):
    if not f.lower().endswith((".png", ".jpg", ".jpeg", ".webp", ".bmp")):
        print(f"Deleting non-image file: {f}")
        os.remove(os.path.join("upload", f))

In [ ]:
start = time.time()

!python inference_realesrgan.py \
    -n RealESRGAN_x4plus \
    -i upload \
    --outscale 3.5 \
    --tile 1200

end = time.time()
print(f"\n Done in {(end - start)/60:.2f} minutes")

In [ ]:
def display(img1, img2, title):
    fig = plt.figure(figsize=(20, 8))
    ax1 = fig.add_subplot(1, 2, 1)
    ax2 = fig.add_subplot(1, 2, 2)
    ax1.imshow(img1)
    ax1.set_title("Original", fontsize=16)
    ax2.imshow(img2)
    ax2.set_title("Upscaled", fontsize=16)
    ax1.axis("off")
    ax2.axis("off")
    plt.suptitle(title, fontsize=18)
    plt.show()

def imread(path):
    img = cv2.imread(path)
    return cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

input_folder = 'upload'
result_folder = 'results'

input_list = sorted([f for f in glob.glob(os.path.join(input_folder, '*.png'))])
output_list = sorted([f for f in glob.glob(os.path.join(result_folder, '*.png'))])


N = 5
for i, (input_path, output_path) in enumerate(zip(input_list, output_list)):
    if i >= N:
        break
    display(imread(input_path), imread(output_path), title=f'Frame {i}')


In [ ]:
results_dir = 'results'
output_video_path = 'upscaled_video4K.mp4'
original_video_path = next((f for f in os.listdir('upload') if f.endswith('.mp4')), None)

frame_paths = sorted(glob.glob(os.path.join(results_dir, '*.png')))
if not frame_paths:
    raise RuntimeError(" No upscaled frames found in 'results/'.")

frame_example = cv2.imread(frame_paths[0])
height, width, _ = frame_example.shape


fps = 25
fourcc = cv2.VideoWriter_fourcc(*'mp4v')
video_writer = cv2.VideoWriter(output_video_path, fourcc, fps, (width, height))

print(f"Writing {len(frame_paths)} frames to video...")
for path in frame_paths:
    frame = cv2.imread(path)
    video_writer.write(frame)

video_writer.release()
print(f" Saved video: {output_video_path}")

if original_video_path:
    print(" Adding audio from original video...")
    original_clip = VideoFileClip(os.path.join('upload', original_video_path))
    upscaled_clip = VideoFileClip(output_video_path).set_audio(original_clip.audio)
    upscaled_clip.write_videofile("upscaled_video_with_audio.mp4", codec="libx264", audio_codec="aac")
    final_output = "upscaled_video_with_audio.mp4"
else:
    print(" No original .mp4 found in upload/ — skipping audio")
    final_output = output_video_path


In [ ]:
print("upload/ contains:")
print(os.listdir('upload'))